# Chapter 3: Tools and Function Calling

In [ ]:
# Setup
from dotenv import load_dotenv
load_dotenv()

%load_ext autoreload
%autoreload 2

## 3.1 Tool Definition Schema

In [ ]:
import json

# Define a calculator tool manually using the OpenAI function calling format
calculator_tool = {
    "type": "function",
    "function": {
        "name": "calculator",
        "description": "Evaluate a mathematical expression and return the result.",
        "parameters": {
            "type": "object",
            "properties": {
                "expression": {
                    "type": "string",
                    "description": "The mathematical expression to evaluate (e.g., '2 + 3 * 4')"
                }
            },
            "required": ["expression"]
        }
    }
}

print(json.dumps(calculator_tool, indent=2))

## 3.2 Automated Tool Definition

In [ ]:
from scratch_agent.tools.helpers import function_to_tool_definition

def calculator(expression: str) -> str:
    """Evaluate a mathematical expression and return the result."""
    try:
        result = eval(expression)
        return str(result)
    except Exception as e:
        return f"Error: {e}"

# Automatically generate the tool definition from the function
tool_def = function_to_tool_definition(calculator)
print(json.dumps(tool_def, indent=2))

## 3.3 Tool Execution

In [ ]:
from litellm import completion
from scratch_agent.tools.helpers import tool_execution

# Build a tool_box mapping tool names to callables
tool_box = {"calculator": calculator}

# Ask the LLM a math question with tools available
messages = [{"role": "user", "content": "What is 1234 * 5678?"}]
response = completion(
    model="gpt-4o-mini",
    messages=messages,
    tools=[tool_def],
)

# If the LLM made a tool call, execute it
choice = response.choices[0]
if choice.message.tool_calls:
    tool_call = choice.message.tool_calls[0]
    print(f"Tool called: {tool_call.function.name}")
    print(f"Arguments: {tool_call.function.arguments}")

    result = tool_execution(tool_box, tool_call)
    print(f"Result: {result}")

    # Send result back to the LLM
    messages.append(choice.message.model_dump())
    messages.append({"role": "tool", "tool_call_id": tool_call.id, "content": result})

    final_response = completion(model="gpt-4o-mini", messages=messages)
    print(f"\nFinal answer: {final_response.choices[0].message.content}")
else:
    print(choice.message.content)

## 3.4 Web Search Tool

In [ ]:
from scratch_agent.tools.search import search_web
from scratch_agent.tools.helpers import function_to_tool_definition

# Generate tool definition for search_web
search_tool_def = function_to_tool_definition(search_web)
print(json.dumps(search_tool_def, indent=2))

# Test the search tool directly
results = search_web("latest developments in AI agents 2025", max_results=3)
for r in results:
    print(f"- {r['title']}: {r['url']}")

## 3.5 MCP Integration

In [ ]:
from scratch_agent.tools.mcp import load_mcp_tools, mcp_tools_to_openai_format

# Define an MCP server connection (example with filesystem server)
connection = {
    "command": "npx",
    "args": ["-y", "@modelcontextprotocol/server-filesystem", "/tmp"]
}

# Load tools from MCP server
mcp_tools = await load_mcp_tools(connection)
print(f"Loaded {len(mcp_tools)} MCP tools:")
for tool in mcp_tools:
    print(f"  - {tool.name}: {tool.description}")

# Convert to OpenAI format for use with LiteLLM
openai_tools = mcp_tools_to_openai_format(mcp_tools)
print(f"\nConverted {len(openai_tools)} tools to OpenAI format")

## 3.6 Simple Agent Loop

In [ ]:
import json
from litellm import completion

def simple_agent_loop(
    user_input: str,
    tools: list[dict],
    tool_box: dict,
    model: str = "gpt-4o-mini",
    system_prompt: str = "You are a helpful assistant.",
    max_steps: int = 5,
) -> str:
    """A simple agent loop that iterates between LLM calls and tool execution.

    Args:
        user_input: The user's question or request.
        tools: List of tool definitions in OpenAI format.
        tool_box: Dict mapping tool names to callables.
        model: The LLM model to use.
        system_prompt: System prompt for the LLM.
        max_steps: Maximum number of tool-calling iterations.

    Returns:
        The final text response from the LLM.
    """
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_input},
    ]

    for step in range(max_steps):
        response = completion(model=model, messages=messages, tools=tools)
        choice = response.choices[0]

        # If no tool calls, we have our final answer
        if not choice.message.tool_calls:
            return choice.message.content

        # Add assistant message with tool calls
        messages.append(choice.message.model_dump())

        # Execute each tool call
        for tool_call in choice.message.tool_calls:
            func_name = tool_call.function.name
            args = json.loads(tool_call.function.arguments)
            print(f"  Step {step + 1}: {func_name}({args})")

            if func_name in tool_box:
                result = str(tool_box[func_name](**args))
            else:
                result = f"Error: Unknown tool '{func_name}'"

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": result,
            })

    return "Max steps reached without final answer."

## 3.7 Running the Simple Agent

In [ ]:
from scratch_agent.tools.helpers import function_to_tool_definition
from scratch_agent.tools.search import search_web

def calculator(expression: str) -> str:
    """Evaluate a mathematical expression and return the result."""
    try:
        return str(eval(expression))
    except Exception as e:
        return f"Error: {e}"

# Set up tools and tool_box
tools = [
    function_to_tool_definition(calculator),
    function_to_tool_definition(search_web),
]
tool_box = {
    "calculator": calculator,
    "search_web": search_web,
}

# Run the agent on a multi-step problem
answer = simple_agent_loop(
    user_input="What is the population of Tokyo? Multiply it by 2 and tell me the result.",
    tools=tools,
    tool_box=tool_box,
)
print(f"\nFinal answer: {answer}")